In [0]:
df = spark.read.table("ecommerce_analytics.bronze.sales_orders")
df.display()


In [0]:
df.printSchema()

In [0]:
... Clixked Item
customer_id | product_id           |Score
19476252    | AVpfPEx61cnluZ0-gyT9 | 34
19476252    | AVpfuJ4pilAPnD_xhDyM | 98
19476252    | AVpe6jFBilAPnD_xQxO2 | 60 
19476252    | AVpfIODe1cnluZ0-eg35 |49
...





...
_** Array of Array **_
...
 [
    ["AVpfPEx61cnluZ0-gyT9","34"],
    ["AVpfuJ4pilAPnD_xhDyM","98"],
    ["AVpe6jFBilAPnD_xQxO2","60"],
    ["AVpfIODe1cnluZ0-eg35","49"]
  ]
...

** Array of Struct **

...
[
    {"curr":"USD","id":"AVpfuJ4pilAPnD_xhDyM","name":"Rony LBT-GPX555 Mini-System with Bluetooth and NFC","price":"993","promotion_info":null,"qty":"3","unit":"pcs"},
    {"curr":"USD","id":"AVpe6jFBilAPnD_xQxO2","name":"Aeon 71.5 x 130.9 16:9 Fixed Frame Projection Screen with CineWhite Projection Surface","price":"218","promotion_info":null,"qty":"3","unit":"pcs"},
    {"curr":"USD","id":"AVpfIODe1cnluZ0-eg35","name":"Cyber-shot DSC-WX220 Digital Camera (Black)","price":"448","promotion_info":null,"qty":"2","unit":"pcs"}
]
...

** ->Array of Struct **

...
[
    {"promo_disc":0.03,"promo_id":"0","promo_item":"AVpfMVD-ilAPnD_xW6bu","promo_qty":"2"}
]
...


In [0]:
df.display()

In [0]:
df.select("ordered_products").display()

In [0]:
[
    {"curr":"USD","id":"AVpfuJ4pilAPnD_xhDyM","name":"Rony LBT-GPX555 Mini-System with Bluetooth and NFC","price":"993","promotion_info":null,"qty":"3","unit":"pcs"},

    {"curr":"USD","id":"AVpe6jFBilAPnD_xQxO2","name":"Aeon 71.5 x 130.9 16:9 Fixed Frame Projection Screen with CineWhite Projection Surface","price":"218","promotion_info":null,"qty":"3","unit":"pcs"},

    {"curr":"USD","id":"AVpfIODe1cnluZ0-eg35","name":"Cyber-shot DSC-WX220 Digital Camera (Black)","price":"448","promotion_info":null,"qty":"2","unit":"pcs"}

 ]

...
          -> StructType and Structfield -> ArrayType
...




In [0]:
from pyspark.sql.types import ArrayType, StructType, StructField, StringType
from pyspark.sql.functions import from_json, col
order_product_schema = ArrayType(
    StructType([
        StructField("curr", StringType()),
        StructField("id", StringType()),
        StructField("name", StringType()),
        StructField("price", StringType()),
        StructField("promotion_info", StringType()),
        StructField("qty", StringType()),
        StructField("unit", StringType())
        ])
    )

df_parsed = df.withColumn("ordered_products", from_json(col("ordered_products"), order_product_schema))

In [0]:
df_parsed.printSchema()

In [0]:
from pyspark.sql.functions import explode_outer

df_exploded_op = df_parsed.withColumn("ordered_products", explode_outer("ordered_products"))

df_exploded_op.display()


In [0]:
df_ordred_products =df_exploded_op.select(
    "customer_id",
    "customer_name",
    "order_number",
    "ordered_products.id",
    col("ordered_products.name"). alias ("product_name"),
    "ordered_products.price",
    "ordered_products.qty",
    "ordered_products.unit",
    "ordered_products.curr"
    )
    
df_ordred_products.display()

In [0]:
df.select("promo_info").display()

In [0]:
[
    {"promo_disc":0.03,"promo_id":"0","promo_item":"AVpfMVD-ilAPnD_xW6bu","promo_qty":"7"}
]

In [0]:
from pyspark.sql.types import*
from pyspark.sql.functions import from_json, col

promo_info_schema = ArrayType(
    StructType([
        StructField("promo_disc", StringType()),
        StructField("promo_id", StringType()),
        StructField("promo_item", StringType()),
        StructField("promo_qty", StringType())
    ])
)

df_promo_parsed = df.withColumn("promo_info", from_json(col("promo_info"), promo_info_schema))

In [0]:
df_promo_parsed.printSchema()

In [0]:
from pyspark.sql.functions import explode_outer

df_exploded_pi = df_promo_parsed.withColumn("promo_info", explode_outer("promo_info")).filter(col("promo_info").isNotNull())

df_exploded_pi.display()



In [0]:
df_promo_parsed = df_exploded_pi.select(
    "customer_id",
    "customer_name",
    "order_number",
    "promo_info.promo_id",
    "promo_info.promo_item",
    col("promo_info.promo_qty").alias("promo_quantity"),
    col("promo_info.promo_disc").alias("promo_discount")
    )

df_promo_parsed.display()

In [0]:
df.select("clicked_items").display()

In [0]:
[
    ["AVpfPEx61cnluZ0-gyT9","34"],
    ["AVpfuJ4pilAPnD_xhDyM","98"],
    ["AVpe6jFBilAPnD_xQxO2","60"],
    ["AVpfIODe1cnluZ0-eg35","49"]
    
]

In [0]:
from pyspark.sql.types import*
from pyspark.sql.functions import from_json, col

clicked_Item_schema = ArrayType(
        StructType([
            StructField("id", StringType()),
            StructField("qty", StringType())
        ])
    )

df_promo_parsed = df.withColumn("clicked_items", from_json(col("clicked_items"), clicked_Item_schema))
df
    

)